In [1]:
import json
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
sys.path.append(os.path.abspath(".."))
from scraper import scrap_website, scrap_website_links

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

In [3]:
links = scrap_website_links("https://www.adventurealtitudetreks.com")
links

['https://wa.me/9779845449032',
 'https://wa.me/61423765587',
 'https://adventurealtitudetreks.com/cart',
 'https://adventurealtitudetreks.com/tailor-made-trip',
 '/',
 'https://adventurealtitudetreks.com/nepal',
 'https://adventurealtitudetreks.com/peak-climbing-in-nepal',
 'https://adventurealtitudetreks.com/island-peak-climbing',
 'https://adventurealtitudetreks.com/lobuche-peak-climbing',
 'https://adventurealtitudetreks.com/island-peak-climbing-from-chhukung',
 'https://adventurealtitudetreks.com/rafting-in-nepal',
 'https://adventurealtitudetreks.com/trishuli-river-rafting',
 'https://adventurealtitudetreks.com/trekking-in-nepal',
 'https://adventurealtitudetreks.com/budget-trek-in-nepal',
 'https://adventurealtitudetreks.com/everest-region-trek',
 'https://adventurealtitudetreks.com/annapurna-region-trek',
 'https://adventurealtitudetreks.com/manaslu-region-trek',
 'https://adventurealtitudetreks.com/langtang-region-trek',
 'https://adventurealtitudetreks.com/mustang-region-trek

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = scrap_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://www.adventurealtitudetreks.com"))


Here is the list of links on the website https://www.adventurealtitudetreks.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://wa.me/9779845449032
https://wa.me/61423765587
https://adventurealtitudetreks.com/cart
https://adventurealtitudetreks.com/tailor-made-trip
/
https://adventurealtitudetreks.com/nepal
https://adventurealtitudetreks.com/peak-climbing-in-nepal
https://adventurealtitudetreks.com/island-peak-climbing
https://adventurealtitudetreks.com/lobuche-peak-climbing
https://adventurealtitudetreks.com/island-peak-climbing-from-chhukung
https://adventurealtitudetreks.com/rafting-in-nepal
https://adventurealtitudetreks.com/trishuli-river-rafting
https://adventurealtitudetreks.com/trekking-in-nepal
https://adventurealtitudetreks.com/budget-trek-in-nepal
https://adventurealtitudetreks.com/eve

In [7]:
def select_relevant_links(url):
    openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [8]:
select_relevant_links("https://www.adventurealtitudetreks.com")

{'links': [{'type': 'about page',
   'url': 'https://adventurealtitudetreks.com/about-us'},
  {'type': 'company page',
   'url': 'https://adventurealtitudetreks.com/company'},
  {'type': 'why choose us page',
   'url': 'https://adventurealtitudes.com/why-choose-us'},
  {'type': 'contact us page',
   'url': 'https://adventure-altitude-trips.surfhost.co.uk/index.php/contact'},
  {'type': 'blog',
   'url': 'https://www.pinterest.com/adventurealtitudetreks/'}],
 'client reviews': [{'type': 'client review', 'reviewId': 1, 'rating': 5},
  {'type': 'client review', 'reviewId': 2, 'rating': 4}]}

### TO MAKE BROCHURE

In [9]:
def fetch_page_and_all_relevant_links(url):
    contents = scrap_website(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += scrap_website(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.adventurealtitudetreks.com"))

In [10]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [11]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [12]:
get_brochure_user_prompt("Adventure Altitude Treks and Expedition", "https://www.adventurealtitudetreks.com")

'\nYou are looking at a company called: Adventure Altitude Treks and Expedition\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nTrekking Agency in Kathmandu | Best Trek Organizer Near Me\n\nTourism License: 2751\n+977 9845449032 (Rohit)\n,\n+61 423 765 587 (Raju)\n0\nPlanning a Trip\nNepal\nPeak Climbing in Nepal\nIsland Peak Climbing- 13 Days\nLobuche Peak Climbing cost For 2025-2026\nIsland Peak Climbing From Chhukung - 3 Day\nRafting In Nepal\nTrishuli River Rafting\nTrekking in Nepal\nBudget Trekking In Nepal\nEverest Region Trek\nAnnapurna Region Trek\nManaslu Region Trek\nLangtang Region Trek\nMustang Region Trek\nDolpha Region Trek\nOther Region\nHeli Tour in Nepal\nEverest Base Camp Helicopter Tour\nEverest View Heli Tour\nLangtang Helicopter Tour\nEverest Base Camp Helicopter Flight Landing Tour\nTours in Nepal\nKathmandu Tour\nManang Mo

In [13]:
def create_brochure(company_name, url):
    openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [14]:
create_brochure("Adventure Altitude Treks", "https://www.adventurealtitudetreks.com")

# Welcome to Adventure Altitude Treks!

Are you ready for the adventure of a lifetime? Look no further than Adventure Altitude Treks, your premier trekking, rafting and tour operator in Nepal and Bhutan.

## Our Culture

We're not just about exploring; we're about connecting with nature, people, and ourselves. Our team is passionate about sharing the beauty and essence of our destinations with every client. We pride ourselves on our warm Nepali hospitality, ensuring you feel at home from day one. When you join us, you become part of a global family that shares values of respect, trust, and shared experiences.

## Meet Your Guides

Our team has spent years honing their skills in the mountains, forests, and valleys of Nepal and Bhutan. You'll be paired with an expert guide who's local born-and-educated, sharing stories, traditions, and unparalleled knowledge about your destination.

## What We Offer

From serene hiking trails to thrilling rafting experiences, island peak climbing adventures, stunning helicopter tours, and private jet travel to some of the most remote sacred sites – we've got it all! Whether you're celebrating a milestone, in search of spiritual enlightenment, or just ready for your next epic adventure:

- **Budget-Friendly**: Affordable packages that suit every budget.
- **Ultimate Escapes**. Exclusive trips tailored just for you and your crew.
- **Adventure Packages**, handpicked to satisfy even the most thrill-seeking souls.

Whether it's rafting down roaring rivers, touching sacred mountains, exploring the mysterious forests of Bhutan, or discovering the timeless beauty hidden within Nepal’s remote corners – Adventure Altitude Treks' mission is to take you where others can only dream.